In [1]:
import numpy as np
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset

trajs = load_dataset('data/plecs_physical_train.npz')

# Check what the inductor voltage looks like in practice
L_true = 100e-6
R_true = 0.1

for i, td in enumerate(trajs[:3]):
    iL = td['i_L']
    VC = td['V_C']
    alpha = td['alpha']
    Vin = td['V_in']
    io = td['i_o']
    
    # Inductor voltage — what drives di_L/dt
    V_L = alpha * Vin - R_true * iL - VC
    
    # What L does the data imply at each timestep?
    t = td['t']
    diL_dt = np.diff(iL) / np.diff(t)
    V_L_mid = 0.5 * (V_L[:-1] + V_L[1:])
    
    # Only where V_L is significant
    mask = np.abs(V_L_mid) > 0.1
    
    if mask.sum() > 0:
        L_implied = diL_dt[mask] / V_L_mid[mask]
        print(f"Traj {i}: {mask.sum()}/{len(mask)} active points")
        print(f"  implied L = {np.median(L_implied)*1e6:.1f} µH "
              f"(std={np.std(L_implied)*1e6:.1f} µH)")
        print(f"  V_L range: [{V_L.min():.3f}, {V_L.max():.3f}] V")
        print(f"  di_L/dt range: [{diL_dt.min():.1f}, {diL_dt.max():.1f}] A/s")
    else:
        print(f"Traj {i}: NO active points — L completely unidentifiable")
    print()

Traj 0: 848/853 active points
  implied L = 93317913.8 µH (std=4131068762.7 µH)
  V_L range: [-3.822, 6.011] V
  di_L/dt range: [-42510.5, 45546.1] A/s

Traj 1: 714/718 active points
  implied L = 40362290.8 µH (std=3583943736.6 µH)
  V_L range: [-2.696, 5.372] V
  di_L/dt range: [-32024.4, 39100.2] A/s

Traj 2: 724/733 active points
  implied L = 69002362.7 µH (std=3272896753.3 µH)
  V_L range: [-4.041, 4.626] V
  di_L/dt range: [-35086.7, 40731.5] A/s



In [2]:
import numpy as np
import torch
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset
from phnn import SelfIdentifyingBuckPHNN

trajs = load_dataset('data/plecs_physical_train.npz')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def compute_rollout_loss(L_val, C_val, R_val, trajs, device):
    model = SelfIdentifyingBuckPHNN(
        initial_L=L_val, initial_C=C_val, initial_R=R_val
    ).to(device)
    # Freeze all params — just evaluate at these exact values
    for p in model.parameters():
        p.requires_grad_(False)
    
    total_loss = 0.0
    total_n = 0
    dt = 50e-6
    
    for td in trajs[:20]:
        iL = td['i_L'].astype(np.float32)
        VC = td['V_C'].astype(np.float32)
        alpha = td['alpha'].astype(np.float32)
        Vin = td['V_in'].astype(np.float32)
        io = td['i_o'].astype(np.float32)
        
        x = torch.tensor(np.stack([iL[:-1], VC[:-1]], axis=1), device=device)
        u = torch.tensor(np.stack([alpha[:-1]*Vin[:-1], io[:-1]], axis=1), device=device)
        xn = torch.tensor(np.stack([iL[1:], VC[1:]], axis=1), device=device)
        
        with torch.no_grad():
            k1 = model(x, u)
            k2 = model(x + 0.5*dt*k1, u)
            k3 = model(x + 0.5*dt*k2, u)
            k4 = model(x + dt*k3, u)
            pred = x + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
            loss = float(((pred - xn)**2).mean())
        
        total_loss += loss
        total_n += 1
    
    return total_loss / total_n

print("Scanning loss landscape over L values:")
print(f"{'L [µH]':>10}  {'loss':>12}")
for L_test in [50, 75, 100, 125, 150, 175, 188, 200]:
    loss = compute_rollout_loss(L_test*1e-6, 100e-6, 0.1, trajs, device)
    print(f"{L_test:>10}  {loss:12.6e}")

Scanning loss landscape over L values:
    L [µH]          loss


IndexError: index 2 is out of bounds for dimension 1 with size 2

In [ ]:
import numpy as np
import torch
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset
from phnn import SelfIdentifyingBuckPHNN

trajs = load_dataset('data/plecs_physical_train.npz')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DT = 50e-6
N = 20

def multistep_loss(L_val, C_val, R_val):
    model = SelfIdentifyingBuckPHNN(L_val, C_val, R_val).to(device)
    for p in model.parameters():
        p.requires_grad_(False)
    
    total, count = 0.0, 0
    for td in trajs[:20]:
        iL = td['i_L'].astype(np.float32)
        VC = td['V_C'].astype(np.float32)
        alpha = td['alpha'].astype(np.float32)
        Vin = td['V_in'].astype(np.float32)
        io = td['i_o'].astype(np.float32)
        x = np.stack([iL, VC], axis=1)
        u = np.stack([alpha*Vin, io], axis=1)
        T = len(iL)
        for start in range(0, T - N - 1, N//2):
            x0 = torch.tensor(x[start][None], device=device)
            u_seq = torch.tensor(u[start:start+N][None], device=device)
            xn = torch.tensor(x[start+1:start+N+1][None], device=device)
            with torch.no_grad():
                xc = x0
                preds = []
                for k in range(N):
                    uk = u_seq[:, k, :]
                    k1 = model(xc, uk)
                    k2 = model(xc + 0.5*DT*k1, uk)
                    k3 = model(xc + 0.5*DT*k2, uk)
                    k4 = model(xc + DT*k3, uk)
                    xc = xc + (DT/6)*(k1+2*k2+2*k3+k4)
                    preds.append(xc)
                pred = torch.stack(preds, dim=1)
                total += float(((pred - xn)**2).mean())
                count += 1
    return total / count

# Scan L×C product (resonant frequency) vs R/L (damping)
print("L×C product scan (fixed R=0.1):")
for L in [80, 90, 100, 110, 120, 133]:
    for C in [70, 80, 90, 100, 110]:
        loss = multistep_loss(L*1e-6, C*1e-6, 0.1)
        lc = L*C
        print(f"  L={L:3d}µH C={C:3d}µF LC={lc:6d}  loss={loss:.4f}")

print("\nR scan (fixed L=100µH, C=100µF):")
for R in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
    loss = multistep_loss(100e-6, 100e-6, R)
    print(f"  R={R:.2f}Ω  loss={loss:.4f}")

L×C product scan (fixed R=0.1):
  L= 80µH C= 70µF LC=  5600  loss=0.7222
  L= 80µH C= 80µF LC=  6400  loss=0.6081
  L= 80µH C= 90µF LC=  7200  loss=0.5027
  L= 80µH C=100µF LC=  8000  loss=0.4233
  L= 80µH C=110µF LC=  8800  loss=0.3726
  L= 90µH C= 70µF LC=  6300  loss=0.6299
  L= 90µH C= 80µF LC=  7200  loss=0.5028
  L= 90µH C= 90µF LC=  8100  loss=0.4099
  L= 90µH C=100µF LC=  9000  loss=0.3550
  L= 90µH C=110µF LC=  9900  loss=0.3334
  L=100µH C= 70µF LC=  7000  loss=0.5347
  L=100µH C= 80µF LC=  8000  loss=0.4184
  L=100µH C= 90µF LC=  9000  loss=0.3514
  L=100µH C=100µF LC= 10000  loss=0.3260
  L=100µH C=110µF LC= 11000  loss=0.3325
  L=110µH C= 70µF LC=  7700  loss=0.4535
  L=110µH C= 80µF LC=  8800  loss=0.3620
  L=110µH C= 90µF LC=  9900  loss=0.3248
  L=110µH C=100µF LC= 11000  loss=0.3275
  L=110µH C=110µF LC= 12100  loss=0.3574
  L=120µH C= 70µF LC=  8400  loss=0.3936
  L=120µH C= 80µF LC=  9600  loss=0.3322
  L=120µH C= 90µF LC= 10800  loss=0.3237
  L=120µH C=100µF LC= 120

In [ ]:
import sys
import importlib
import inspect
from pathlib import Path

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

import phnn
print(phnn.__file__)

importlib.reload(phnn)
from phnn import SelfIdentifyingBuckPHNN

print(inspect.signature(SelfIdentifyingBuckPHNN.__init__))


c:\Users\rcper\OneDrive\Desktop\Research\Research2026\DCtoDC Converter Characterization with Deep Learning\Characterizing Buck Converter withPort Hamiltonian Neural Network\src\phnn.py
(self, initial_L: 'float' = 0.0001, initial_C: 'float' = 0.0001, initial_R: 'float' = 0.1, initial_R_C: 'float' = 0.05, r_epsilon: 'float' = 1e-06) -> 'None'


In [ ]:
import numpy as np
import torch
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset
from phnn import SelfIdentifyingBuckPHNN

trajs = load_dataset('data/plecs_physical_train.npz')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DT = 50e-6
N = 20

def multistep_loss_4param(L_val, C_val, R_L_val, R_C_val):
    model = SelfIdentifyingBuckPHNN(
        initial_L=L_val, initial_C=C_val,
        initial_R=R_L_val, initial_R_C=R_C_val
    ).to(device)
    for p in model.parameters():
        p.requires_grad_(False)
    
    total, count = 0.0, 0
    for td in trajs[:20]:
        iL = td['i_L'].astype(np.float32)
        VC = td['V_C'].astype(np.float32)
        alpha = td['alpha'].astype(np.float32)
        Vin = td['V_in'].astype(np.float32)
        io = td['i_o'].astype(np.float32)
        x = np.stack([iL, VC], axis=1)
        u = np.stack([alpha*Vin, io], axis=1)
        T = len(iL)
        for start in range(0, T - N - 1, N//2):
            x0 = torch.tensor(x[start][None], device=device)
            u_seq = torch.tensor(u[start:start+N][None], device=device)
            xn = torch.tensor(x[start+1:start+N+1][None], device=device)
            with torch.no_grad():
                xc = x0
                preds = []
                for k in range(N):
                    uk = u_seq[:, k, :]
                    k1 = model(xc, uk)
                    k2 = model(xc + 0.5*DT*k1, uk)
                    k3 = model(xc + 0.5*DT*k2, uk)
                    k4 = model(xc + DT*k3, uk)
                    xc = xc + (DT/6)*(k1+2*k2+2*k3+k4)
                    preds.append(xc)
                pred = torch.stack(preds, dim=1)
                total += float(((pred - xn)**2).mean())
                count += 1
    return total / count

# First: what does the true 4-parameter point give?
print("True parameters:")
print(f"  L=100, C=100, R_L=0.10, R_C=0.05: {multistep_loss_4param(100e-6, 100e-6, 0.10, 0.05):.4f}")
print(f"  Learned: L=131, C=86,  R_L=0.20, R_C=0.14: {multistep_loss_4param(131e-6, 86e-6, 0.20, 0.14):.4f}")

# Scan total effective resistance at true L, C
print("\nR_L + R_C scan (fixed L=100µH, C=100µF):")
for R_total in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    # Split evenly between R_L and R_C
    loss = multistep_loss_4param(100e-6, 100e-6, R_total*0.67, R_total*0.33)
    print(f"  R_total={R_total:.2f} (R_L={R_total*0.67:.3f}, R_C={R_total*0.33:.3f}): {loss:.4f}")

# What alpha range does your data cover?
print("\nAlpha distribution in dataset:")
alphas = [td['alpha'][0] for td in trajs]
print(f"  min={min(alphas):.2f}, max={max(alphas):.2f}, mean={np.mean(alphas):.2f}")

True parameters:
  L=100, C=100, R_L=0.10, R_C=0.05: 0.2982
  Learned: L=131, C=86,  R_L=0.20, R_C=0.14: 0.2909

R_L + R_C scan (fixed L=100µH, C=100µF):
  R_total=0.10 (R_L=0.067, R_C=0.033): 0.3198
  R_total=0.15 (R_L=0.101, R_C=0.050): 0.2983
  R_total=0.20 (R_L=0.134, R_C=0.066): 0.2881
  R_total=0.25 (R_L=0.168, R_C=0.083): 0.2853
  R_total=0.30 (R_L=0.201, R_C=0.099): 0.2875
  R_total=0.35 (R_L=0.234, R_C=0.115): 0.2931
  R_total=0.40 (R_L=0.268, R_C=0.132): 0.3010

Alpha distribution in dataset:
  min=0.21, max=0.79, mean=0.47


In [ ]:
import numpy as np
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset

trajs = load_dataset('data/plecs_physical_train.npz')

diL_list, VL_list, dVC_list, icap_list, iL_list = [], [], [], [], []

for td in trajs:
    iL    = td['i_L']
    VC    = td['V_C']
    alpha = td['alpha']
    Vin   = td['V_in']
    io    = td['i_o']
    t     = td['t']
    dt    = np.median(np.diff(t))

    diL_dt = np.diff(iL) / dt
    dVC_dt = np.diff(VC) / dt

    iL_mid    = 0.5*(iL[:-1]  + iL[1:])
    VC_mid    = 0.5*(VC[:-1]  + VC[1:])
    alpha_mid = 0.5*(alpha[:-1] + alpha[1:])
    Vin_mid   = 0.5*(Vin[:-1]  + Vin[1:])
    io_mid    = 0.5*(io[:-1]   + io[1:])

    i_cap = iL_mid - io_mid
    aVin  = alpha_mid * Vin_mid

    diL_list.append(diL_dt)
    VL_list.append(aVin - VC_mid)
    dVC_list.append(dVC_dt)
    icap_list.append(i_cap)
    iL_list.append(iL_mid)

diL  = np.concatenate(diL_list)
VL0  = np.concatenate(VL_list)
dVC  = np.concatenate(dVC_list)
icap = np.concatenate(icap_list)
iL_all = np.concatenate(iL_list)

mask = np.abs(VL0) > 0.2
print(f"Active points: {mask.sum()}/{len(mask)} ({100*mask.sum()/len(mask):.1f}%)")

diL_m  = diL[mask]
VL0_m  = VL0[mask]
dVC_m  = dVC[mask]
icap_m = icap[mask]
iL_m   = iL_all[mask]

# Least squares fit of di_L/dt = inv_L*(VL0 - R_L*iL - R_C*icap)
A = np.column_stack([VL0_m, -iL_m, -icap_m])
coeffs, _, _, _ = np.linalg.lstsq(A, diL_m, rcond=None)
inv_L, inv_L_RL, inv_L_RC = coeffs
L_fit  = 1.0 / inv_L
RL_fit = inv_L_RL / inv_L
RC_fit = inv_L_RC / inv_L

print(f"\nLeast-squares fit of inductor equation:")
print(f"  L   = {L_fit*1e6:.2f} µH  (true: 100 µH)")
print(f"  R_L = {RL_fit:.4f} Ω    (true: 0.10 Ω)")
print(f"  R_C = {RC_fit:.4f} Ω    (true: 0.05 Ω)")

# Least squares fit of dV_C/dt = inv_C * icap
inv_C_fit = np.dot(icap_m, dVC_m) / np.dot(icap_m, icap_m)
C_fit = 1.0 / inv_C_fit
print(f"\nLeast-squares fit of capacitor equation:")
print(f"  C   = {C_fit*1e6:.2f} µF  (true: 100 µF)")

# Residuals
diL_pred = inv_L * (VL0_m - RL_fit*iL_m - RC_fit*icap_m)
dVC_pred = inv_C_fit * icap_m
print(f"\nResidual RMS (how well the averaged model fits the data):")
print(f"  di_L/dt: {np.sqrt(np.mean((diL_m - diL_pred)**2)):.1f} A/s")
print(f"  dV_C/dt: {np.sqrt(np.mean((dVC_m - dVC_pred)**2)):.1f} V/s")
print(f"  vs typical magnitudes:")
print(f"  |di_L/dt|: {np.sqrt(np.mean(diL_m**2)):.1f} A/s")
print(f"  |dV_C/dt|: {np.sqrt(np.mean(dVC_m**2)):.1f} V/s")

Active points: 32421/46962 (69.0%)

Least-squares fit of inductor equation:
  L   = 208.46 µH  (true: 100 µH)
  R_L = 0.2625 Ω    (true: 0.10 Ω)
  R_C = -0.2251 Ω    (true: 0.05 Ω)

Least-squares fit of capacitor equation:
  C   = 98.17 µF  (true: 100 µF)

Residual RMS (how well the averaged model fits the data):
  di_L/dt: 4750.9 A/s
  dV_C/dt: 1451.6 V/s
  vs typical magnitudes:
  |di_L/dt|: 6583.8 A/s
  |dV_C/dt|: 6481.7 V/s


In [ ]:
import numpy as np
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset

trajs = load_dataset('data/plecs_physical_train.npz')

# Check: does i_o match V_C / R_load_implied at steady state?
# And does i_L match i_o at steady state?
print(f"{'Traj':>4} {'alpha':>6} {'V_in':>6} {'aVin':>6} {'V_C_ss':>7} {'i_L_ss':>7} {'i_o_ss':>7} {'V_C/i_o':>8} {'aVin-V_C':>9}")
for i in [64, 11, 14, 13]:
    if i >= len(trajs):
        continue
    td = trajs[i]
    # Take last 100 points as steady state
    n = len(td['t'])
    s = slice(max(0, n-100), n)
    iL_ss = float(np.mean(td['i_L'][s]))
    VC_ss = float(np.mean(td['V_C'][s]))
    io_ss = float(np.mean(td['i_o'][s]))
    a = float(td['alpha'][0])
    Vin = float(td['V_in'][0])
    aVin = a * Vin
    R_load_impl = VC_ss / io_ss if io_ss > 0.01 else 0
    print(f"{i:>4} {a:>6.2f} {Vin:>6.2f} {aVin:>6.2f} {VC_ss:>7.3f} {iL_ss:>7.3f} {io_ss:>7.3f} {R_load_impl:>8.2f} {aVin-VC_ss:>9.3f}")

# Also check: what does the steady-state PHYSICS say V_C should be?
# At ss: i_L = i_o, V_C = aVin - R_L*i_L (if R_C contribution to ss is zero since i_cap=0 at ss)
# So V_C_ss should equal aVin - 0.10 * i_o_ss
print(f"\nPhysics prediction: V_C_ss = aVin - 0.10 * i_o_ss")
for i in [64, 11, 14, 13]:
    if i >= len(trajs):
        continue
    td = trajs[i]
    n = len(td['t'])
    s = slice(max(0, n-100), n)
    iL_ss = float(np.mean(td['i_L'][s]))
    VC_ss = float(np.mean(td['V_C'][s]))
    io_ss = float(np.mean(td['i_o'][s]))
    a = float(td['alpha'][0])
    Vin = float(td['V_in'][0])
    aVin = a * Vin
    VC_predicted = aVin - 0.10 * io_ss
    print(f"  Traj {i}: predicted V_C = {VC_predicted:.3f}V, actual = {VC_ss:.3f}V, diff = {VC_ss - VC_predicted:+.3f}V")

Traj  alpha   V_in   aVin  V_C_ss  i_L_ss  i_o_ss  V_C/i_o  aVin-V_C
  64   0.60  11.43   6.81   7.081   0.768   0.768     9.22    -0.266
  11   0.47  10.07   4.76   5.601   0.747   0.746     7.51    -0.840
  14   0.76  13.64  10.38   9.077   0.625   0.628    14.46     1.307
  13   0.36  12.29   4.37   4.238   0.228   0.226    18.74     0.130

Physics prediction: V_C_ss = aVin - 0.10 * i_o_ss
  Traj 64: predicted V_C = 6.738V, actual = 7.081V, diff = +0.343V
  Traj 11: predicted V_C = 4.686V, actual = 5.601V, diff = +0.915V
  Traj 14: predicted V_C = 10.321V, actual = 9.077V, diff = -1.244V
  Traj 13: predicted V_C = 4.346V, actual = 4.238V, diff = -0.108V


In [ ]:
import numpy as np
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset

trajs = load_dataset('data/plecs_physical_train.npz')

print(f"{'#':>3} {'alpha':>6} {'V_in':>6} {'aVin':>6} {'V_C_ss':>7} {'i_L_ss':>7} {'aVin-VC':>8} {'violation':>10}")
print("-" * 70)

violations = 0
for i, td in enumerate(trajs):
    a = float(td['alpha'][0])
    Vin = float(td['V_in'][0])
    aVin = a * Vin
    
    # Get steady state from end of each segment
    io = td['i_o']
    t = td['t']
    iL = td['i_L']
    VC = td['V_C']
    
    boundaries = list(np.where(np.abs(np.diff(io)) > 0.05)[0])
    ends = boundaries + [len(io)-1]
    
    for e in ends:
        ss = slice(max(0, e-20), e+1)
        VC_ss = float(np.mean(VC[ss]))
        iL_ss = float(np.mean(iL[ss]))
        gap = aVin - VC_ss
        if VC_ss > aVin + 0.05:
            violations += 1
            print(f"{i:>3} {a:>6.2f} {Vin:>6.2f} {aVin:>6.2f} {VC_ss:>7.3f} {iL_ss:>7.3f} {gap:>+8.3f} {'VIOLATION':>10}")

print(f"\nTotal violations: {violations}")

  #  alpha   V_in   aVin  V_C_ss  i_L_ss  aVin-VC  violation
----------------------------------------------------------------------
  2   0.41  11.14   4.60   4.744   2.457   -0.142  VIOLATION
  2   0.41  11.14   4.60   5.017   1.983   -0.415  VIOLATION
  2   0.41  11.14   4.60   5.086   1.537   -0.484  VIOLATION
  2   0.41  11.14   4.60   5.002   1.178   -0.400  VIOLATION
  2   0.41  11.14   4.60   4.840   0.942   -0.238  VIOLATION
  2   0.41  11.14   4.60   4.798   1.079   -0.196  VIOLATION
  2   0.41  11.14   4.60   5.080   1.014   -0.478  VIOLATION
  2   0.41  11.14   4.60   5.284   0.845   -0.682  VIOLATION
  2   0.41  11.14   4.60   5.380   0.608   -0.778  VIOLATION
  2   0.41  11.14   4.60   4.818   0.091   -0.216  VIOLATION
  2   0.41  11.14   4.60   4.915   0.500   -0.313  VIOLATION
  2   0.41  11.14   4.60   4.936   0.268   -0.334  VIOLATION
  2   0.41  11.14   4.60   4.902   0.357   -0.300  VIOLATION
  3   0.46  12.03   5.57   5.696   1.752   -0.126  VIOLATION
  3   0.46  12

In [3]:
import numpy as np
import sys
sys.path.insert(0, 'src')
from generate_dataset import load_dataset

trajs = load_dataset('data/plecs_physical_train.npz')

print("Checking TRUE steady state (last 20 samples of full trajectory only):")
print(f"{'#':>3} {'alpha':>6} {'V_in':>6} {'aVin':>6} {'V_C_ss':>7} {'i_L_ss':>7} {'diff':>8} {'settled?':>10}")
print("-" * 65)

violations = 0
for i, td in enumerate(trajs):
    a = float(td['alpha'][0])
    Vin = float(td['V_in'][0])
    aVin = a * Vin
    
    # Only look at the very end of the full trajectory
    n = len(td['t'])
    ss = slice(n-20, n)
    VC_ss = float(np.mean(td['V_C'][ss]))
    iL_ss = float(np.mean(td['i_L'][ss]))
    
    # Check if i_L is actually stable (not still changing)
    iL_std = float(np.std(td['i_L'][ss]))
    settled = iL_std < 0.05  # stable if std < 50mA over last 20 samples
    
    diff = aVin - VC_ss
    if VC_ss > aVin + 0.1:
        violations += 1
        print(f"{i:>3} {a:>6.2f} {Vin:>6.2f} {aVin:>6.2f} {VC_ss:>7.3f} {iL_ss:>7.3f} {diff:>+8.3f} {'SETTLED' if settled else 'TRANSIENT':>10}")

print(f"\nViolations at true steady state: {violations}/{len(trajs)}")

Checking TRUE steady state (last 20 samples of full trajectory only):
  #  alpha   V_in   aVin  V_C_ss  i_L_ss     diff   settled?
-----------------------------------------------------------------
  2   0.41  11.14   4.60   4.904   0.356   -0.302    SETTLED
  4   0.72  11.51   8.28   8.518   1.105   -0.238    SETTLED
 11   0.75  10.37   7.80   8.953   0.498   -1.156    SETTLED
 12   0.52  10.40   5.44   6.245   0.494   -0.806    SETTLED
 14   0.46  10.85   4.97   5.453   0.461   -0.483    SETTLED
 17   0.66  10.70   7.10   7.916   0.396   -0.817    SETTLED
 19   0.26  10.62   2.75   3.088   0.167   -0.339    SETTLED
 20   0.80  10.01   7.97   9.448   1.017   -1.479    SETTLED
 28   0.67  10.60   7.15   8.021   0.681   -0.869    SETTLED
 32   0.51  10.51   5.38   6.108   0.400   -0.726    SETTLED
 35   0.20  10.96   2.23   2.396   0.375   -0.168    SETTLED
 37   0.49  11.41   5.64   5.901   0.307   -0.264    SETTLED
 41   0.64  10.84   6.96   7.674   0.404   -0.718    SETTLED
 45   0.70

In [ ]:
import numpy as np
import sys
from pathlib import Path
sys.path.insert(0, 'src')
from generate_plecs_dataset import PLECSRPCSimulator

model_path = Path('models/buck_converter.plecs')
sim = PLECSRPCSimulator(model_path)

# Use the exact same params as your generation script
params = {
    'V_in': 10.37,
    'L':    100e-6,
    'C':    100e-6,
    'ESR_L': 0.1,
    'ESR_C': 0.0,
    'R_load': 15.0,
    'alpha': 0.75,
    'cycle_frequency': 100e3,
}

t_end = 5e-3  # just 5ms — enough to see steady state

raw = sim.simulate(params, t_end)

t_raw = np.asarray(raw['Time'], dtype=float) - raw['Time'][0]
iL_raw = np.asarray(raw['Values'][0], dtype=float)
VC_raw = np.asarray(raw['Values'][1], dtype=float)

print(f"Total raw points: {len(t_raw)}")
print(f"dt range: [{np.diff(t_raw).min()*1e9:.1f}, {np.diff(t_raw).max()*1e9:.1f}] ns")
print(f"alpha*V_in = {0.75*10.37:.3f} V")
print(f"V_C raw range: [{VC_raw.min():.3f}, {VC_raw.max():.3f}] V")
print(f"i_L raw range: [{iL_raw.min():.3f}, {iL_raw.max():.3f}] A")

# Look at last 5 switching cycles
T_sw = 1e-5
for cycle in range(5, 0, -1):
    ta = t_raw[-1] - cycle * T_sw
    tb = ta + T_sw
    mask = (t_raw >= ta) & (t_raw <= tb)
    if mask.sum() < 2:
        continue
    t_c = t_raw[mask]
    VC_c = VC_raw[mask]
    iL_c = iL_raw[mask]
    VC_avg = np.trapz(VC_c, t_c) / (t_c[-1] - t_c[0])
    iL_avg = np.trapz(iL_c, t_c) / (t_c[-1] - t_c[0])
    print(f"Cycle -{cycle}: {mask.sum():3d} pts | "
          f"V_C=[{VC_c.min():.3f},{VC_c.max():.3f}] avg={VC_avg:.3f}V | "
          f"i_L avg={iL_avg:.3f}A")

Loaded buck_converter
Total raw points: 10001
dt range: [500.0, 500.0] ns
alpha*V_in = 7.777 V
V_C raw range: [0.000, 15.807] V
i_L raw range: [-5.545, 8.552] A
Cycle -5:  20 pts | V_C=[8.838,8.846] avg=8.841V | i_L avg=0.504A
Cycle -4:  21 pts | V_C=[8.830,8.838] avg=8.833V | i_L avg=0.509A
Cycle -3:  20 pts | V_C=[8.823,8.829] avg=8.825V | i_L avg=0.527A
Cycle -2:  21 pts | V_C=[8.818,8.823] avg=8.820V | i_L avg=0.533A
Cycle -1:  20 pts | V_C=[8.814,8.817] avg=8.815V | i_L avg=0.552A


C:\Users\rcper\AppData\Local\Temp\ipykernel_37640\3089385426.py:47: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  VC_avg = np.trapz(VC_c, t_c) / (t_c[-1] - t_c[0])
C:\Users\rcper\AppData\Local\Temp\ipykernel_37640\3089385426.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  iL_avg = np.trapz(iL_c, t_c) / (t_c[-1] - t_c[0])


: 